# RFE Expanded


This notebook extends `rfe.ipynb` by applying the feature subset selected by RFE to all baseline models for comparison.


In [ ]:
import sys
from pathlib import Path

_here = Path().resolve()
REPO_ROOT = _here
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "src" / "stroke_data.py").is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError("Could not find src/stroke_data.py (open this project from the repo folder).")

sys.path.insert(0, str(REPO_ROOT / "src"))


## Imports


In [ ]:
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from stroke_data import get_stroke_data_for_cv


## Data And Configuration


In [ ]:
CSV = "data/knn-standardize-distance.csv"
OUT = "src/feature_selection/rfe_selection.csv"
N_FEATURES_TO_SELECT = 8
RFE_STEP = 1

csv_path = REPO_ROOT / CSV
df = pd.read_csv(csv_path)
feature_names = [c for c in df.columns if c not in ("id", "stroke")]

X_train, X_test, y_train, y_test = get_stroke_data_for_cv(str(csv_path))
len(feature_names), X_train.shape


## Run RFE


In [ ]:
base_estimator = LogisticRegression(
    random_state=42,
    class_weight="balanced",
    max_iter=2000,
    solver="lbfgs",
)

selector = RFE(
    estimator=base_estimator,
    n_features_to_select=N_FEATURES_TO_SELECT,
    step=RFE_STEP,
)
selector.fit(X_train, y_train)

rfe_df = pd.DataFrame({
    "feature": feature_names,
    "selected": selector.support_,
    "ranking": selector.ranking_,
}).sort_values(["selected", "ranking", "feature"], ascending=[False, True, True])

rfe_df


## Save RFE Selection


In [ ]:
out_path = REPO_ROOT / OUT
out_path.parent.mkdir(parents=True, exist_ok=True)
rfe_df.to_csv(out_path, index=False)
out_path


## Build Selected Subset


In [ ]:
mask = selector.support_
selected_features = rfe_df.loc[rfe_df["selected"], "feature"].tolist()

X_train_subset = X_train[:, mask]
X_test_subset = X_test[:, mask]

selected_features


## Baseline Models


In [ ]:
def build_baseline_models():
    return {
        "lr": LogisticRegression(random_state=42, C=0.1, class_weight="balanced", solver="newton-cg"),
        "svm": SVC(C=10.0, class_weight="balanced", gamma=0.01, kernel="rbf"),
        "rf": RandomForestClassifier(
            bootstrap=True,
            class_weight="balanced_subsample",
            max_depth=15,
            max_features=None,
            max_leaf_nodes=15,
            n_estimators=20,
            random_state=42,
        ),
        "xgb": XGBClassifier(
            eta=1,
            gamma=1,
            reg_lambda=0.5,
            max_depth=15,
            objective="binary:logistic",
            subsample=1,
            eval_metric="logloss",
        ),
        "gnb": GaussianNB(priors=None, var_smoothing=1e-09),
        "knn": KNeighborsClassifier(algorithm="auto", leaf_size=10, metric="minkowski", n_neighbors=5, weights="uniform"),
    }


def metric_row(model_name, subset_name, Xtr, Xte):
    model = clone(build_baseline_models()[model_name])
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    return {
        "model": model_name,
        "subset": subset_name,
        "n_features": Xtr.shape[1],
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred, pos_label=1),
        "precision": precision_score(y_test, pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_test, pred, pos_label=1, zero_division=0),
    }


## Evaluate Full vs RFE Subset Across All Baselines


In [ ]:
subset_data = {
    "full": (X_train, X_test),
    "rfe-subset": (X_train_subset, X_test_subset),
}

rows = []
for model_name in build_baseline_models():
    for subset_name, (Xtr, Xte) in subset_data.items():
        rows.append(metric_row(model_name, subset_name, Xtr, Xte))

eval_df = pd.DataFrame(rows).sort_values(["model", "n_features"]).reset_index(drop=True)
eval_df.style.format({
    "accuracy": "{:.4f}",
    "f1": "{:.4f}",
    "precision": "{:.4f}",
    "recall": "{:.4f}",
}).hide(axis="index")


## Pivot View For Easier Comparison


In [ ]:
eval_df.pivot(index="model", columns="subset", values=["accuracy", "f1", "precision", "recall"]).round(4)


## Optional Export


In [ ]:
expanded_out = REPO_ROOT / "src/feature_selection/rfe_expand_results.csv"
eval_df.to_csv(expanded_out, index=False)
expanded_out
